# Should contain all preprocessing that independent of run configs

### Imports

In [ ]:
import numpy as np 
import polars as pl 
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from numba import njit  
import math 
from collections import namedtuple
import json5

In [2]:
import utils as ut
import a_preprocessing as a
import b_run_staging as b

### Loading data dicts and initial preprocessing

In [ ]:
## Loading in data and configs
input_dir = '/home/ma/a/alb25/Project/thesis_code/data/processed/intermediate/input'
input_files = [f'{input_dir}/train_df.parquet',f'{input_dir}/validation_df.parquet',f'{input_dir}/test_df.parquet',]

df = pl.scan_parquet(input_files).select(['source_user@domain', 'time'])
static_configs = ut.load_json5('static_configs')

### Getting helper dictionaries
train_test_dict = a.get_train_test_split()
bin_metric_dict = a.get_bin_metrics()
bin_metric_dict = a.add_training_denom(bin_metric_dict)
ut.dump_json5(bin_metric_dict, 'bin_metric_dict')
ut.dump_json5(train_test_dict, 'train_test_dict')

### Preprocessing data
user_counts = a.create_counts_data(df, bin_metric_dict)
user_counts, user_mapping = a.create_user_to_id_mapping(user_counts, mapping_file_name='source_users_to_id_mapping')
user_counts = a.create_coarse_bins(user_counts, bin_metric_dict)
user_interactions = a.create_first_last_interaction_arrays(user_counts=user_counts)


SchemaError: extra column in file outside of expected schema: destination_user@domain, hint: specify this column in the schema, or pass extra_columns='ignore' in scan options

### Getting Grids and interpolation weights

### Getting interpolation weights + Storing outputs

In [ ]:
# Creating initial grids
u_init, v_init = b.init_grid_NB(user_counts, n_users = user_mapping.shape[0], coarse_bins_per_week = bin_metric_dict['coarse_bins_per_week'], 
                              period_start=train_test_dict['train_start'], period_end=train_test_dict['train_end'], bin_metric_dict=bin_metric_dict)

u_pos_init, v_pos_init, p_init = b.init_grid_hurdle(user_counts, n_users = user_mapping.shape[0], coarse_bins_per_week = bin_metric_dict['coarse_bins_per_week'], 
                                                    period_start=train_test_dict['train_start'], period_end=train_test_dict['train_end'], bin_metric_dict=bin_metric_dict)

# Creating the grids used for clustering
u_clustering, v_clustering = b.init_grid_NB(user_counts, n_users=user_mapping.shape[0], coarse_bins_per_week=bin_metric_dict['coarse_bins_per_week'], 
                                            period_start=train_test_dict['train_start'], period_end=train_test_dict['burn_in_end'], bin_metric_dict=bin_metric_dict)

u_pos_clustering, v_pos_clustering, p_pos_clustering = b.init_grid_hurdle(user_counts, n_users=user_mapping.shape[0], coarse_bins_per_week=bin_metric_dict['coarse_bins_per_week'], 
                                            period_start=train_test_dict['train_start'], period_end=train_test_dict['burn_in_end'], bin_metric_dict=bin_metric_dict)

# Creating initial n_counts_grid
n_counts_init = b.init_n_counts_grid(user_counts, n_users=user_mapping.shape[0], coarse_bins_per_week=bin_metric_dict['coarse_bins_per_week'], 
                                     period_start=train_test_dict['train_start'], period_end=train_test_dict['train_end'])

# Getting the degen mask
degen_mask = b.get_degen_mask(user_counts, n_users=user_mapping.shape[0], n_coarse_bins=bin_metric_dict['coarse_bins_per_week'], static_configs=static_configs, train_test_dict=train_test_dict)

# Getting interpolation weights
interpolation_weights = b.get_linear_interpolation_weights(bin_metric_dict)

In [ ]:
## Preparing outputs for storgae
degen_bins_per_user = user_mapping.with_columns(pl.Series('n_degen_bins', degen_mask.astype('int64').sum(axis=1)))
user_counts = user_counts.select(['user_id', 'fine_bin_id', 'count'])

# Storing outputs
ut.store_data(degen_bins_per_user, 'degen_bins_per_user')
ut.store_data(user_interactions, 'user_interactions')
ut.store_data(user_counts, 'user_counts')
ut.store_data(interpolation_weights, filename='interpolation_weights')
ut.store_data(u_init, 'u_init')
ut.store_data(v_init, 'v_init')
ut.store_data(p_init, 'p_init')
ut.store_data(u_pos_init, 'u_pos_init')
ut.store_data(v_pos_init, 'v_pos_init')
ut.store_data(n_counts_init, 'n_counts_init')
ut.store_data(user_mapping, "user_mapping")
ut.store_data(degen_mask, "degen_mask")
ut.store_data(u_clustering, "u_clustering")
ut.store_data(v_clustering, "v_clustering")
ut.store_data(u_pos_clustering, "u_pos_clustering")
ut.store_data(v_pos_clustering, "v_pos_clustering")
ut.store_data(p_pos_clustering, "p_pos_clustering")